# 15 — Attribution: Explaining a Prediction

**Purpose:** The "why did the model say that?" workflow — `torchlens.attribution`'s
eight methods, the `AttributionResult` container, target specification, and the
heatmap-rendering tie-in with `tl.viz`.

**Surfaces covered:**
- [ ] `torchlens.attribution.saliency(model, x, target=...)` — gradient magnitude
- [ ] `torchlens.attribution.input_x_grad` — input * gradient
- [ ] `torchlens.attribution.integrated_gradients(..., n_steps=, baseline=)`
- [ ] `torchlens.attribution.smoothgrad(..., n_samples=, noise_level=, seed=)`
- [ ] `torchlens.attribution.grad_cam(..., layer=, relu=)` — conv-layer CAM
- [ ] `torchlens.attribution.layer_attribution(..., layer=, method=)`
- [ ] `torchlens.attribution.layer_conductance(..., layer=)`
- [ ] `torchlens.attribution.layer_integrated_gradients(..., layer=)`
- [ ] `AttributionResult` — repr, `.method`, `.values`, `.target_repr`, `.extra`
- [ ] `AttributionError` — the failure surface (bad layer / bad target)
- [ ] `tl.viz.render_heatmap` / `tl.viz.causal_trace_heatmap` on attribution scores

## 1. Environment setup

In [ ]:
import pathlib
import sys
import warnings

# Pin imports to THIS checkout: the notebook dir (for _models.py) plus the repo
# root, so `import torchlens` audits the code this notebook ships with -- not a
# pip-installed copy from another checkout.
_NB_DIR = pathlib.Path.cwd()
if not (_NB_DIR / "_models.py").exists():
    _NB_DIR = next(
        p
        for p in [_NB_DIR / "notebooks" / "audit", *_NB_DIR.parents]
        if (p / "_models.py").exists()
    )
_REPO_ROOT = _NB_DIR.parents[1]
sys.path.insert(0, str(_NB_DIR))
sys.path.insert(0, str(_REPO_ROOT))

warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings as _w

_w.filterwarnings("ignore", message=".*pynvml.*")  # torch/cuda probe noise
import torch
import torchlens as tl

assert pathlib.Path(tl.__file__).is_relative_to(_REPO_ROOT), (
    f"torchlens imported from {tl.__file__} -- expected this checkout"
)
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

attr = tl.attribution  # reachable directly as a tl attribute
print("tl.attribution:", attr.__name__)

## 2. The core workflow — `saliency` and `AttributionResult`

Every method shares one calling convention: `method(model, inputs, target=...)` with
keyword-only options, returning an `AttributionResult`. `target` picks which output
scalar to explain (an int class index for a `[batch, classes]` output).

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()  # 8 -> relu -> 4 "classes"

result = attr.saliency(model, x, target=0)

print("type        :", type(result).__name__)
print("repr        :")
print(repr(result))
print()
print("result.method     :", result.method)
print("result.target_repr:", result.target_repr)
print("result.values     :", type(result.values).__name__, tuple(result.values.shape))
print("result.extra      :", result.extra)

## 3. The gradient family — `input_x_grad`, `integrated_gradients`, `smoothgrad`

Same convention throughout; each has its own knobs (`n_steps`, `baseline`,
`n_samples`/`noise_level`/`seed`). We compare the per-feature rankings the four input
methods produce for the same prediction.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

results = {
    "saliency": attr.saliency(model, x, target=0),
    "input_x_grad": attr.input_x_grad(model, x, target=0),
    "integrated_gradients": attr.integrated_gradients(model, x, target=0, n_steps=16),
    "smoothgrad": attr.smoothgrad(model, x, target=0, n_samples=10, noise_level=0.1, seed=0),
}

print(f"{'method':<22} {'values shape':<14} top-3 input features (sample 0)")
for name, res in results.items():
    scores = res.values[0].abs()
    top3 = scores.topk(3).indices.tolist()
    print(f"{name:<22} {str(tuple(res.values.shape)):<14} {top3}")

## 4. Layer methods — `layer_attribution`, `layer_conductance`, `layer_integrated_gradients`

These explain a prediction in terms of a NAMED internal layer instead of the input.
`layer=` takes a module address (as in `trace.modules`).

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

# Module addresses for tiny_mlp: 'in_proj', 'out_proj'
for fn, kwargs in [
    (attr.layer_attribution, dict(method="activation_x_grad")),
    (attr.layer_conductance, dict(n_steps=8)),
    (attr.layer_integrated_gradients, dict(n_steps=8)),
]:
    try:
        res = fn(model, x, target=0, layer="in_proj", **kwargs)
        print(f"{fn.__name__:<28} -> values {tuple(res.values.shape)}  method={res.method!r}")
    except Exception as exc:
        print(f"⚠️ {fn.__name__:<26} -> {type(exc).__name__}: {str(exc)[:90]}")

## 5. `grad_cam` — class activation mapping on a conv layer

`grad_cam` wants a convolutional layer address; `relu=True` (default) keeps only
positive evidence. `tiny_branch_cnn` provides real conv activations.

In [ ]:
torch.manual_seed(0)
model_cnn, x_cnn = ZOO["tiny_branch_cnn"]()

# Find a conv module address from a quick trace
trace_cnn = tl.trace(model_cnn, x_cnn)
conv_addrs = [k for k in trace_cnn.modules.keys() if ":" not in k and "conv" in k.lower()]
print("conv module addresses:", conv_addrs)

try:
    cam = attr.grad_cam(model_cnn, x_cnn, target=0, layer=conv_addrs[0])
    print()
    print("grad_cam ->", repr(cam)[:160])
    print("cam.values shape:", tuple(cam.values.shape), " (spatial evidence map)")
except Exception as exc:
    print(f"⚠️ grad_cam -> {type(exc).__name__}: {str(exc)[:120]}")

## 6. Rendering attribution — the `tl.viz` tie-in

`AttributionResult.values` is a plain tensor, so the `tl.viz` direct renderers apply:
`render_heatmap` for a quick look, `causal_trace_heatmap` for the
mechanistic-interpretability styling (score grid with outlier clipping).

In [ ]:
sal = results["saliency"].values

hm = tl.viz.render_heatmap(sal, width=240, height=80)
print("render_heatmap(saliency.values) ->", type(hm).__name__, hm.size)

try:
    cth = tl.viz.causal_trace_heatmap(sal)
    print("causal_trace_heatmap(saliency.values) ->", type(cth).__name__, getattr(cth, "size", ""))
except Exception as exc:
    print(f"⚠️ causal_trace_heatmap -> {type(exc).__name__}: {str(exc)[:120]}")

## 7. The failure surface — `AttributionError` and bad targets

What a user sees when they typo a layer address or pass a nonsense target.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

# Bad layer address
try:
    attr.layer_attribution(model, x, target=0, layer="does_not_exist")
    print("bad layer: no error raised (GAP)")
except attr.AttributionError as exc:
    print(f"bad layer -> AttributionError: {str(exc)[:110]}")
except Exception as exc:
    print(f"bad layer -> {type(exc).__name__} (not AttributionError): {str(exc)[:110]}")

# Out-of-range target index
try:
    attr.saliency(model, x, target=99)
    print("target=99: no error raised (GAP)")
except attr.AttributionError as exc:
    print(f"target=99 -> AttributionError: {str(exc)[:110]}")
except Exception as exc:
    print(f"target=99 -> {type(exc).__name__} (not AttributionError): {str(exc)[:110]}")

---

## ⚠️ GAPs / ergonomic smells

- **`tl.attribution` is a live attribute but appears in no `__all__`, docs example, or
  top-level name list** — a user has no discovery path to eight shipped methods short
  of reading the source tree (this was flagged as "shipped-but-invisible" in the
  2026-07-05 feature inventory; still true at the notebook level until now).
- **`AttributionResult.__repr__` inlines the full values tensor** — fine on a
  2x8 toy, unusable on a real model (a ResNet CAM would dump thousands of numbers).
  A shape-summary repr (`AttributionResult(method='saliency', values=<2x8 tensor>,
  target=0)`) would match the rest of the library's repr discipline.
- **`AttributionError` messages are genuinely good** (bad layer suggests available
  layers; bad target names the output shape) — positive finding worth keeping.
- **`render_heatmap` (PIL Image) vs `causal_trace_heatmap` (matplotlib Axes +
  side-effect Figure)** — two heatmap renderers in one namespace with divergent
  return types and display behavior.
- The attribution methods run the model themselves (fresh forward+backward per call);
  there is no way to reuse an existing `Trace`, so explaining k targets costs k
  forwards. A `trace=`-accepting variant would fit the TorchLens model better.